In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
import sys
from tqdm.auto import tqdm
from antropy import perm_entropy

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, lowpass_filter_causal, bootstrap_report

/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sampling_rate = 100 # Hz, sensor sampling frequency
lpf_cutoff = 10 # Hz, FIR lowpass cutoff
lpf_order = 64 # FIR filter taps
pe_order = 5 # permutation entropy embedding dimension
pe_delay = int(sampling_rate / (2 * lpf_cutoff)) # PE delay (Nyquist-matched)
alpha = 0.023 # false-alarm bound
glue = False # set True to glue metrics for result collection
per_ride_path = None # set a path to save per-ride outcomes as CSV

In [3]:
# Parameters
alpha = 0.1
glue = True
per_ride_path = None


In [4]:
threshold = 1 - alpha # non-sober probability decision threshold

In [5]:
data = pd.read_parquet("../data/data.parquet")
data

,ride_id,time_index,participant_id,run,trial,ax,ay,az,rx,ry,rz,throttle,brake_l,brake_r
0,000e5ab725f4,0,23.0,Sober,2.0,-0.442073,-9.215956,-3.105285,0.020311,-0.136070,0.213650,41.0,41.0,41.0
1,000e5ab725f4,10,23.0,Sober,2.0,-0.547358,-9.390632,-2.988635,0.019853,-0.149967,0.210443,41.0,41.0,41.0
2,000e5ab725f4,20,23.0,Sober,2.0,-0.552742,-9.348757,-2.974876,0.017715,-0.165086,0.208457,41.0,41.0,41.0
3,000e5ab725f4,30,23.0,Sober,2.0,-0.605384,-9.287740,-3.065803,0.014508,-0.182190,0.206319,41.0,41.0,41.0
4,000e5ab725f4,40,23.0,Sober,2.0,-0.631106,-9.226125,-3.071786,0.014966,-0.197767,0.204181,41.0,41.0,41.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876191,fdc39ec37c87,67410,13.0,Sober,2.0,1.475773,-10.374680,-0.763908,-0.276263,1.067486,0.098043,147.0,41.0,41.0
876192,fdc39ec37c87,67420,13.0,Sober,2.0,1.175474,-9.505487,-1.128814,-0.290924,1.067792,0.093462,147.0,41.0,41.0
876193,fdc39ec37c87,67430,13.0,Sober,2.0,1.273579,-10.332806,-0.802791,-0.313373,1.102611,0.105374,147.0,41.0,41.0
876194,fdc39ec37c87,67440,13.0,Sober,2.0,1.231107,-9.077770,-1.895115,-0.332616,1.102306,0.106443,147.0,41.0,41.0


In [6]:
labels = data.groupby(["ride_id", "participant_id"])["run"].first().reset_index()
labels

,ride_id,participant_id,run
0,000e5ab725f4,23.0,Sober
1,02a49cf28afc,31.0,High
2,04a58ca54c11,28.0,Sober
3,065ea8ad5f24,23.0,Low
4,097169d2138f,10.0,Low
...,...,...,...
136,fbf0250f1611,8.0,High
137,fc09a596ffcd,25.0,High
138,fcb8375a05a2,24.0,Low
139,fd0f1af171a2,23.0,Low


In [7]:
features = ["ax", "ay", "az", "rx", "ry", "rz", "throttle"]

data_np = reshape_to_numpy(
    data,
    features = features,
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_lpf = lowpass_filter_causal(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=lpf_cutoff,
    gyro_cutoff=lpf_cutoff,
    order=lpf_order,
    fs=sampling_rate,
)

In [8]:
# Compute one permutation-entropy value per channel per ride.
rows = []
for ride in tqdm(data_np_lpf, desc="entropy"):
    ent = []
    for ch in ride.T:
        ch = ch[~np.isnan(ch)] # strip FIR warm-up and trailing NaNs
        pe = perm_entropy(ch, order=pe_order, delay=pe_delay, normalize=True)
        ent.append(pe)
    rows.append(ent)

# convert to numpy
X_feat = np.array(rows)
X_feat.shape

entropy:   0%|                                                                                                                                                                  | 0/141 [00:00<?, ?it/s]

entropy:  13%|████████████████████▍                                                                                                                                   | 19/141 [00:00<00:00, 181.20it/s]

entropy:  27%|████████████████████████████████████████▉                                                                                                               | 38/141 [00:00<00:00, 182.79it/s]

entropy:  40%|█████████████████████████████████████████████████████████████▍                                                                                          | 57/141 [00:00<00:00, 180.78it/s]

entropy:  54%|█████████████████████████████████████████████████████████████████████████████████▉                                                                      | 76/141 [00:00<00:00, 179.69it/s]

entropy:  67%|█████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                  | 94/141 [00:00<00:00, 179.36it/s]

entropy:  80%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                              | 113/141 [00:00<00:00, 181.05it/s]

entropy:  94%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎         | 132/141 [00:00<00:00, 183.20it/s]

entropy: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 141/141 [00:00<00:00, 181.64it/s]

(141, 7)

In [9]:
# Within-subject centering (WSC): subtract per-channel mean of the
# participant's other rides to remove between-subject variability
# without needing to know which ride is sober.
X_wsc = np.empty_like(X_feat)
for i in range(len(labels)): # i is a positional index, not a label
    # mask of the same participant's other rides
    pid = labels["participant_id"].iloc[i]
    others = (labels["participant_id"] == pid)
    others.iloc[i] = False
    # subtract their per-channel mean to center within subject
    X_wsc[i] = X_feat[i] - X_feat[others.values].mean(axis=0)

In [10]:
# LOPO CV: fit a 3-class logistic regression per held-out participant
# and collect per-ride class probabilities.
logo = LeaveOneGroupOut()
groups = labels["participant_id"]
y_true_all, y_proba_all, pid_all = [], [], []

for train_idx, test_idx in tqdm(list(logo.split(X_wsc, labels["run"], groups)), desc="LOPO"):
    # fit on training set
    clf = make_pipeline(StandardScaler(), LogisticRegression())
    clf.fit(X_wsc[train_idx], labels["run"].iloc[train_idx])
    # infer on test set and append to results
    y_true_all.extend(labels["run"].iloc[test_idx])
    pid_all.extend(labels["participant_id"].iloc[test_idx])
    y_proba_all.append(clf.predict_proba(X_wsc[test_idx]))

# stack results
y_proba_all = np.vstack(y_proba_all)

LOPO:   0%|                                                                                                                                                                      | 0/25 [00:00<?, ?it/s]

LOPO:  64%|███████████████████████████████████████████████████████████████████████████████████████████████████▊                                                        | 16/25 [00:00<00:00, 159.03it/s]

LOPO: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:00<00:00, 167.08it/s]

In [11]:
# detection rule: flag a ride as impaired if P(non-sober) >= threshold
y_true_arr = np.array(y_true_all) # truth to numpy
sober_col = np.where(clf.classes_ == "Sober")[0][0] # column index of the Sober class
p_non_sober = 1 - y_proba_all[:, sober_col] # non-sober probability per ride
detected = p_non_sober >= threshold # apply detection threshold
df_res = pd.DataFrame({"run": y_true_arr, "participant_id": pid_all, "detected": detected}) # make results 
df_res

,run,participant_id,detected
0,High,3.0,True
1,Sober,3.0,False
2,High,3.0,True
3,Low,3.0,False
4,Low,3.0,False
...,...,...,...
136,Sober,31.0,False
137,Low,31.0,True
138,Low,31.0,False
139,Sober,31.0,False


In [12]:
_ = bootstrap_report(df_res, glue=glue, per_ride_path=per_ride_path)

       support     detection_rate                FAR
run                                                 
High        46  0.98 [0.93, 1.00]                NaN
Low         50  0.56 [0.44, 0.68]                NaN
Sober       45                NaN  0.02 [0.00, 0.07]

accuracy sober vs non-sober: 0.83 [0.78, 0.88]
accuracy sober vs high:      0.98 [0.93, 1.00]


/home/marco/git/online-impairment-detection/.venv/lib/python3.13/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)
